# 1b. Model Parameter Sensitivity (BE & SC)

**Purpose**: Systematic characterisation of how each parameter affects summary
statistics, for both the Boundary Estimation (BE) and Stimulus-Category (SC) models.

**Key outputs**:
- 1D sweeps with confidence bands per model
- Side-by-side comparison: same stats, different models
- Sensitivity matrices for both models
- Pairwise identifiability assessment


## 1. Configuration

In [ ]:
import sys, os
sys.path.append(os.path.abspath(".."))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.gridspec as gridspec
from scipy.stats import spearmanr
from itertools import combinations
import warnings
warnings.filterwarnings('ignore')

# --- BE imports ---
from models.BE_core import BEParams, BEState, BEModel
# --- SC imports ---
from models.SC_core import SCParams, SCState, SCModel
# --- Shared ---
from models.perception import perceive_stimulus, stimulus_space_bounds
from behav_utils.data.synthetic import sample_stimuli
# stimulus_to_category was removed; categories are returned by sample_stimuli.
# If you need it standalone:
def stimulus_to_category(stimuli, boundary=0.0):
    cat = stimuli > boundary
    return cat.astype(int)

from behav_utils.analysis.summary_stats import (
	compute_summary_stats, get_stat_names_expanded, SUMMARY_REGISTRY,
	list_available_stats,
)
from behav_utils.analysis.update_matrix import compute_update_matrix
from behav_utils.plotting.psychometric import plot_psychometric
from behav_utils.plotting.update_matrix import plot_update_matrix

%matplotlib inline
plt.rcParams['figure.dpi'] = 100

In [ ]:
# ── Stats to track ─────────────────────────────────────────────────────────
ALL_STAT_NAMES = [
	'accuracy', 'psychometric', 'recency', 'win_stay', 'win_stay_rate',
	'lose_shift', 'choice_autocorr', 'side_bias', 'stimulus_sensitivity',
	'choice_entropy', 'perseveration', 'logistic_history',
	'hard_easy_ratio', 'hard_accuracy', 'easy_accuracy',
	'conditional_psychometric', 'update_matrix',
	'binned_accuracy', 'binned_choice_prob',
]

expanded = get_stat_names_expanded(ALL_STAT_NAMES)
SCALAR_STATS = [s for s in expanded if not s.startswith('_')]
print(f"Tracking {len(SCALAR_STATS)} scalar stats")

# ── Simulation config ──────────────────────────────────────────────────────
N_TRIALS   = 500
BURN_IN    = 200
N_SEEDS    = 20
N_SWEEP    = 8
SEED_BASE  = 42

# ── BE baseline & sweep ───────────────────────────────────────────────────
BE_BASE = dict(sigma_percep=0.15, A_repulsion=0.10, eta_learning=0.35, eta_relax=0.12)
BE_PARAM_NAMES = list(BE_BASE.keys())
BE_BOUNDS = BEParams.get_bounds()
BE_SWEEP = {p: np.linspace(BE_BOUNDS[p][0] + 0.01, BE_BOUNDS[p][1] - 0.01, N_SWEEP)
			 for p in BE_PARAM_NAMES}

# ── SC baseline & sweep ───────────────────────────────────────────────────
SC_BASE = dict(sigma_percep=0.15, A_repulsion=0.10, gamma=0.95, sigma_update=0.30)
SC_PARAM_NAMES = list(SC_BASE.keys())
SC_BOUNDS = SCParams.get_bounds()
SC_SWEEP = {p: np.linspace(SC_BOUNDS[p][0] + 0.01, SC_BOUNDS[p][1] - 0.01, N_SWEEP)
			 for p in SC_PARAM_NAMES}

## 2. Simulation Helpers

In [ ]:
rng = np.random.default_rng(42)

stimuli = sample_stimuli(100, 'Uniform', rng)
# categories = stimulus_to_category(stimuli)

In [ ]:
def simulate_one_be(params_dict, n_trials=N_TRIALS, burn_in=BURN_IN, seed=42):
	params = BEParams(**params_dict)
	rng = np.random.default_rng(seed)

	state = BEState.initial_uniform()
	if burn_in > 0:
		bi_stim = rng.uniform(-1, 1, burn_in)
		bi_cats = stimulus_to_category(bi_stim)
		_, _, state, _ = BEModel.simulate_session(params, state, bi_stim, bi_cats, rng)

	stimuli, categories = sample_stimuli(n_trials, 'Uniform', rng)
	# categories = stimulus_to_category(stimuli)
	choices, p_B, state_out, trace = BEModel.simulate_session(
		params, state, stimuli, categories, rng, return_history=True,
	)

	stats = compute_summary_stats(choices, stimuli, categories,
								   stat_names=ALL_STAT_NAMES, return_dict=True)
	stats['_trace'] = trace
	stats['_update_matrix'] = compute_update_matrix(stimuli, choices, categories, n_bins=8)
	return stats


def simulate_one_sc(params_dict, n_trials=N_TRIALS, burn_in=BURN_IN, seed=42):
	params = SCParams(**params_dict)
	rng = np.random.default_rng(seed)

	state = SCState.initial_default()
	if burn_in > 0:
		bi_stim = rng.uniform(-1, 1, burn_in)
		bi_cats = stimulus_to_category(bi_stim)
		_, _, state, _ = SCModel.simulate_session(params, state, bi_stim, bi_cats, rng)

	stimuli,categories  = sample_stimuli(n_trials, 'Uniform', rng)
	# categories = stimulus_to_category(stimuli)
	choices, p_B, state_out, trace = SCModel.simulate_session(
		params, state, stimuli, categories, rng, return_history=True,
	)

	stats = compute_summary_stats(choices, stimuli, categories,
								   stat_names=ALL_STAT_NAMES, return_dict=True)
	stats['_trace'] = trace
	stats['_update_matrix'] = compute_update_matrix(stimuli, choices, categories, n_bins=8)
	
	return stats


def simulate_sweep(sim_fn, base_params, param_name, sweep_values, n_seeds=N_SEEDS):
	"""Run sim_fn across sweep values × seeds. Returns {value: DataFrame}."""
	results = {}
	for val in sweep_values:
		rows = []
		p = dict(base_params)
		p[param_name] = float(val)
		for si in range(n_seeds):
			res = sim_fn(p, seed=SEED_BASE + si)
			row = {k: v for k, v in res.items() if not k.startswith('_')}
			rows.append(row)
		results[val] = pd.DataFrame(rows)
	return results

print("Helpers defined.")

## 3. Quick Sanity Check — BE vs SC at Baseline

Before running full sweeps, verify both models produce sensible behaviour.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 4))
fig.suptitle('Baseline Sanity Check', fontsize=13, fontweight='bold')

for model_name, sim_fn, base, colour in [
	('BE', simulate_one_be, BE_BASE, 'steelblue'),
	('SC', simulate_one_sc, SC_BASE, 'darkorange'),
]:
	res = sim_fn(base, seed=42)
	trace = res['_trace']
	um = res['_update_matrix']
	# Psychometric
	ax = axes[0]
	ax.set_title('Psychometric')
	plot_psychometric(trace.stimuli, trace.choices, ax=ax,
					   color=colour, label=model_name)

	# Update matrix (use last model's for shape)
	ax_idx = 1 if model_name == 'BE' else 2
	plot_update_matrix(um[0], ax=axes[ax_idx])
	axes[ax_idx].set_title(f'Update Matrix ({model_name})')

	# Extract values safely
	accuracy = res.get('accuracy', np.nan)
	recency = res.get('recency', np.nan)

	psych = res.get('_update_matrix', [{}]*3)[2].get('total_psychometric', {})
	mu = psych.get('mu', np.nan)
	sigma = psych.get('sigma', np.nan)

	# Define x labels and y values
	x_labels = ['recency', 'accuracy', 'mu', 'sigma']
	y_values = [recency, accuracy, mu, sigma]

	# Convert categorical labels to numeric positions
	x_positions = np.arange(len(x_labels))

	# Scatter plot
	axes[3].scatter(
		x_positions,
		y_values,
		color=colour,
		alpha=0.8,
	)

	# Set x-axis labels
	axes[3].set_xticks(x_positions)
	axes[3].set_xticklabels(x_labels)

	axes[3].set_ylabel("Value")
	axes[3].set_title(f"{model_name} Metrics")

axes[0].legend()
axes[3].set_title('Key Stats')
plt.tight_layout()
plt.show()

# Print summary
for model_name, sim_fn, base in [('BE', simulate_one_be, BE_BASE),
								   ('SC', simulate_one_sc, SC_BASE)]:
	res = sim_fn(base, seed=42)
	print(f"\n{model_name} baseline: accuracy={res.get('accuracy', 0):.3f}, "
		  f"recency={res.get('recency', 0):.3f}, "
		  f"pse={res['_update_matrix'][2]['total_psychometric'].get('mu', np.nan):.3f}," 
		  f"slope={res['_update_matrix'][2]['total_psychometric'].get('sigma', np.nan):.3f}")
	

## 4. 1D Sweeps with Confidence Bands

For each model, sweep each parameter while holding others at baseline.
Each point = N_SEEDS simulations. Lines show median, bands show 25th–75th percentile.

In [ ]:
# ── Run BE sweeps ──────────────────────────────────────────────────────────
print("Running BE sweeps...")
be_sweep_data = {}
for pname in BE_PARAM_NAMES:
	print(f"  {pname}: {N_SWEEP} values × {N_SEEDS} seeds...")
	be_sweep_data[pname] = simulate_sweep(
		simulate_one_be, BE_BASE, pname, BE_SWEEP[pname])
print("BE done.")

# ── Run SC sweeps ──────────────────────────────────────────────────────────
print("\nRunning SC sweeps...")
sc_sweep_data = {}
for pname in SC_PARAM_NAMES:
	print(f"  {pname}: {N_SWEEP} values × {N_SEEDS} seeds...")
	sc_sweep_data[pname] = simulate_sweep(
		simulate_one_sc, SC_BASE, pname, SC_SWEEP[pname])
print("SC done.")

In [ ]:
# ── Key stats to plot ──────────────────────────────────────────────────────
PLOT_STATS = [
	'accuracy', 'recency', 'mu', 'sigma',
	'stimulus_sensitivity', 'choice_entropy',
	'win_stay_rate', 'perseveration',
	'hard_accuracy', 'easy_accuracy',
]
PLOT_STATS = [s for s in PLOT_STATS if s in SCALAR_STATS]

def plot_sweep_comparison(be_data, sc_data, be_values, sc_values,
						   be_pname, sc_pname, stats=PLOT_STATS):
	"""Side-by-side sweep plots for BE and SC."""
	n_stats = len(stats)
	n_cols = min(4, n_stats)
	n_rows = int(np.ceil(n_stats / n_cols))

	fig, axes = plt.subplots(n_rows, n_cols * 2, figsize=(3.5 * n_cols * 2, 3 * n_rows))
	if axes.ndim == 1:
		axes = axes.reshape(1, -1)
	fig.suptitle(f'BE: {be_pname} sweep  |  SC: {sc_pname} sweep',
				 fontsize=13, fontweight='bold', y=1.02)

	for i, sname in enumerate(stats):
		row, col = divmod(i, n_cols)

		# BE (left half)
		ax_be = axes[row, col]
		medians, q25, q75 = [], [], []
		for val in be_values:
			col_data = be_data[val][sname].dropna()
			medians.append(col_data.median())
			q25.append(col_data.quantile(0.25))
			q75.append(col_data.quantile(0.75))
		ax_be.fill_between(be_values, q25, q75, alpha=0.3, color='steelblue')
		ax_be.plot(be_values, medians, 'o-', color='steelblue', ms=3)
		ax_be.set_ylabel(sname, fontsize=9)
		ax_be.set_xlabel(be_pname, fontsize=8)
		ax_be.tick_params(labelsize=7)

		# SC (right half)
		ax_sc = axes[row, col + n_cols]
		medians, q25, q75 = [], [], []
		for val in sc_values:
			col_data = sc_data[val][sname].dropna()
			medians.append(col_data.median())
			q25.append(col_data.quantile(0.25))
			q75.append(col_data.quantile(0.75))
		ax_sc.fill_between(sc_values, q25, q75, alpha=0.3, color='darkorange')
		ax_sc.plot(sc_values, medians, 'o-', color='darkorange', ms=3)
		ax_sc.set_xlabel(sc_pname, fontsize=8)
		ax_sc.tick_params(labelsize=7)

	# Hide empty axes
	for j in range(len(stats), n_rows * n_cols):
		row, col = divmod(j, n_cols)
		axes[row, col].set_visible(False)
		axes[row, col + n_cols].set_visible(False)

	plt.tight_layout()
	plt.show()

In [ ]:
# ── Shared params: sigma_percep ────────────────────────────────────────────
plot_sweep_comparison(
	be_sweep_data['sigma_percep'], sc_sweep_data['sigma_percep'],
	BE_SWEEP['sigma_percep'], SC_SWEEP['sigma_percep'],
	'sigma_percep (BE)', 'sigma_percep (SC)',
)

In [ ]:
# ── Shared params: A_repulsion ─────────────────────────────────────────────
plot_sweep_comparison(
	be_sweep_data['A_repulsion'], sc_sweep_data['A_repulsion'],
	BE_SWEEP['A_repulsion'], SC_SWEEP['A_repulsion'],
	'A_repulsion (BE)', 'A_repulsion (SC)',
)

In [ ]:
# ── Learning rate comparison: eta_learning (BE) vs 1-gamma (SC) ────────────
# These are the "learning rate" parameters — functionally analogous
# BE: higher eta_learning = more updating
# SC: lower gamma (higher 1-gamma) = more updating

n_stats = len(PLOT_STATS)
n_cols = min(4, n_stats)
n_rows = int(np.ceil(n_stats / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 3 * n_rows))
axes_flat = np.array(axes).flatten()
fig.suptitle('Learning rate comparison: eta_learning (BE, blue) vs 1-gamma (SC, orange)',
			 fontsize=12, fontweight='bold', y=1.02)

for i, sname in enumerate(PLOT_STATS):
	ax = axes_flat[i]

	# BE: eta_learning
	be_vals = BE_SWEEP['eta_learning']
	medians_be = [be_sweep_data['eta_learning'][v][sname].median() for v in be_vals]
	ax.plot(be_vals, medians_be, 'o-', color='steelblue', ms=3, label='BE eta')

	# SC: 1-gamma (invert x-axis so both go low→high learning rate)
	sc_vals = SC_SWEEP['gamma']
	sc_lr = 1.0 - sc_vals  # effective learning rate
	medians_sc = [sc_sweep_data['gamma'][v][sname].median() for v in sc_vals]
	ax.plot(sc_lr, medians_sc, 's-', color='darkorange', ms=3, label='SC 1-γ')

	ax.set_ylabel(sname, fontsize=9)
	ax.set_xlabel('effective learning rate', fontsize=8)
	ax.tick_params(labelsize=7)
	if i == 0:
		ax.legend(fontsize=7)

for j in range(len(PLOT_STATS), len(axes_flat)):
	axes_flat[j].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
# ── SC-specific: sigma_update sweep ────────────────────────────────────────
vals = SC_SWEEP['sigma_update']
n_stats = len(PLOT_STATS)
n_cols = min(4, n_stats)
n_rows = int(np.ceil(n_stats / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 3 * n_rows))
axes_flat = np.array(axes).flatten()
fig.suptitle('SC-specific: sigma_update sweep (update bump width)',
			 fontsize=12, fontweight='bold', y=1.02)

for i, sname in enumerate(PLOT_STATS):
	ax = axes_flat[i]
	medians, q25, q75 = [], [], []
	for val in vals:
		col = sc_sweep_data['sigma_update'][val][sname].dropna()
		medians.append(col.median())
		q25.append(col.quantile(0.25))
		q75.append(col.quantile(0.75))
	ax.fill_between(vals, q25, q75, alpha=0.3, color='darkorange')
	ax.plot(vals, medians, 'o-', color='darkorange', ms=3)
	ax.set_ylabel(sname, fontsize=9)
	ax.set_xlabel('sigma_update', fontsize=8)
	ax.tick_params(labelsize=7)

for j in range(len(PLOT_STATS), len(axes_flat)):
	axes_flat[j].set_visible(False)

plt.tight_layout()
plt.show()

## 5. Formal Sensitivity Matrix

Cohen's d across parameter range, normalised by within-param-value variability.

In [ ]:
def compute_sensitivity(sweep_data, param_names, sweep_values, plot_stats):
	"""Compute sensitivity dict for a model."""
	sensitivity = {}
	for pname in param_names:
		values = sweep_values[pname]
		data = sweep_data[pname]
		available = [s for s in plot_stats if s in data[values[0]].columns]

		for sname in available:
			per_val = [data[val][sname].dropna().values for val in values]
			low_vals, high_vals = per_val[0], per_val[-1]
			if len(low_vals) < 3 or len(high_vals) < 3:
				sensitivity[(sname, pname)] = {'d': np.nan, 'cv': np.nan, 'ratio': np.nan}
				continue

			pooled_std = np.sqrt((np.var(low_vals) + np.var(high_vals)) / 2)
			d = (np.mean(high_vals) - np.mean(low_vals)) / pooled_std if pooled_std > 1e-10 else 0

			cvs = []
			for pv in per_val:
				m = np.mean(pv)
				cvs.append(np.std(pv) / abs(m) if abs(m) > 1e-10 else np.nan)
			mean_cv = np.nanmean(cvs)

			sensitivity[(sname, pname)] = {
				'd': d,
				'cv': mean_cv,
				'ratio': abs(d) / mean_cv if mean_cv > 1e-10 else abs(d) * 100,
			}
	return sensitivity


be_sensitivity = compute_sensitivity(be_sweep_data, BE_PARAM_NAMES, BE_SWEEP, PLOT_STATS)
sc_sensitivity = compute_sensitivity(sc_sweep_data, SC_PARAM_NAMES, SC_SWEEP, PLOT_STATS)
print(f"BE: {len(be_sensitivity)} (stat, param) entries")
print(f"SC: {len(sc_sensitivity)} (stat, param) entries")

In [ ]:
# ── Side-by-side sensitivity heatmaps ─────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, max(6, len(PLOT_STATS) * 0.4)))

for ax, sensitivity, param_names, title, cmap in [
	(ax1, be_sensitivity, BE_PARAM_NAMES, 'BE Sensitivity (|d|/CV)', 'Blues'),
	(ax2, sc_sensitivity, SC_PARAM_NAMES, 'SC Sensitivity (|d|/CV)', 'Oranges'),
]:
	avail_stats = [s for s in PLOT_STATS
				   if any((s, p) in sensitivity for p in param_names)]
	mat = np.full((len(avail_stats), len(param_names)), np.nan)
	for i, s in enumerate(avail_stats):
		for j, p in enumerate(param_names):
			mat[i, j] = sensitivity.get((s, p), {}).get('ratio', np.nan)

	im = ax.imshow(mat, cmap=cmap, aspect='auto', vmin=0,
					vmax=np.nanpercentile(mat, 95))
	ax.set_xticks(range(len(param_names)))
	ax.set_xticklabels(param_names, rotation=45, ha='right', fontsize=8)
	ax.set_yticks(range(len(avail_stats)))
	ax.set_yticklabels(avail_stats, fontsize=8)
	ax.set_title(title, fontsize=11, fontweight='bold')
	plt.colorbar(im, ax=ax, shrink=0.8)

plt.tight_layout()
plt.show()

## 6. Summary

**Shared parameters** (sigma_percep, A_repulsion): Should show similar qualitative
effects across both models — these control the same perceptual front-end.

**Learning rate** (eta_learning vs 1-gamma): Functionally analogous but different
mechanisms. BE updates a boundary distribution; SC updates category distributions.
Look for whether recency, accuracy, and PSE respond similarly.

**SC-specific** (sigma_update): Controls locality of the Gaussian update bump.
Wider bumps → more generalisation across stimuli, narrower → more local learning.
No BE equivalent.

**Key questions**:
- Do both models span the same range of behavioural signatures?
- Are any summary stats uniquely informative for one model but not the other?
- Can the same stat set serve for SBI with both models?
